# Première lecture des documents FOMC

Point de départ : `data/raw/fomc_documents_raw.csv`.

In [ ]:
import os
import re
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomc_nlp_matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "fomc_documents_raw.csv").exists():
    ROOT = ROOT.parent

raw_path = ROOT / "data" / "raw" / "fomc_documents_raw.csv"
raw = pd.read_csv(raw_path, parse_dates=["date"])
raw.head(1)

Premier aperçu du fichier chargé.

In [ ]:
pd.Series({
    "rows": len(raw),
    "first_date": raw["date"].min().date(),
    "last_date": raw["date"].max().date(),
    "duplicated_dates": raw["date"].duplicated().sum(),
    "empty_statements": raw["statement_text"].fillna("").str.len().eq(0).sum(),
    "empty_minutes": raw["minutes_text"].fillna("").str.len().eq(0).sum(),
})

199 réunions entre 2000 et 2024. Les deux textes sont présents à chaque fois.

Nettoyage : minuscules, espaces propres, comptage de mots par regex.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", re.IGNORECASE)

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace(" ", " ")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def count_words(text):
    return len(TOKEN_RE.findall(clean_text(text)))

df = raw.assign(**{
    "statement_clean_text": raw["statement_text"].map(clean_text),
    "minutes_clean_text": raw["minutes_text"].map(clean_text),
    "statement_n_words": raw["statement_text"].map(count_words),
    "minutes_n_words": raw["minutes_text"].map(count_words),
})

df[["date", "statement_n_words", "minutes_n_words"]].head()

In [ ]:
lengths = pd.concat([
    df[["date", "year", "statement_n_words"]].rename(columns={"statement_n_words": "n_words"}).assign(document_type="statement"),
    df[["date", "year", "minutes_n_words"]].rename(columns={"minutes_n_words": "n_words"}).assign(document_type="minutes"),
], ignore_index=True)

lengths.groupby("document_type")["n_words"].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=lengths, x="date", y="n_words", hue="document_type", ax=ax)
ax.set_yscale("log")
ax.set_title("Document length")
ax.set_xlabel("Meeting date")
ax.set_ylabel("Words, log scale");

La taille des documents tend à augmenter légèrement dans le temps. Les Minutes sont surtout beaucoup plus longues que les Statements.

Les pics réguliers dans les Minutes viennent en partie des réunions avec projections économiques, qui ajoutent plus de contenu à la publication.


#### Décision de taux

On l'extrait du Statement avec un regex.

In [ ]:
RATE_PATTERNS = {
    "hike": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(raise|raising|increase|increasing)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(raise|raising|increase|increasing|increased)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "cut": [
        r"\b(decided|voted|agreed|approved)\b.{0,80}\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target|federal funds|discount rate)\b",
        r"\b(lower|lowering|reduce|reducing|decrease|decreasing|cut)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
    ],
    "hold": [
        r"\b(decided|voted|agreed)\b.{0,80}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(will|would|shall|to)\b.{0,20}\b(maintain|keep|leave)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(maintain|maintaining|keep|keeping|kept|leave|leaving)\b.{0,80}\b(target range|target for the federal funds rate|federal funds rate)\b",
        r"\b(target range|target for the federal funds rate|federal funds rate)\b.{0,80}\b(unchanged|maintained)\b",
    ],
}

def infer_rate_decision(text):
    text = clean_text(text)
    for label in ["hike", "cut", "hold"]:
        if any(re.search(pattern, text, flags=re.DOTALL) for pattern in RATE_PATTERNS[label]):
            return label
    return "unknown"

df = df.assign(rate_decision=df["statement_text"].map(infer_rate_decision))
display(df["rate_decision"].value_counts(dropna=False))

fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(data=df, x="year", hue="rate_decision", ax=ax)
ax.set_title("Inferred rate decisions over time")
ax.set_xlabel("Meeting year")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Inferred rate decision", loc="upper left", bbox_to_anchor=(1, 1))

## 4. Inflation vs marché du travail

Angle principal : quelle place la Fed donne à l'inflation et au marché du travail ?

In [ ]:
INFLATION_TERMS = [
    "inflation", "inflationary", "prices", "price stability", "price pressures",
    "inflation expectations", "core inflation", "pce inflation", "consumer prices",
    "energy prices", "food prices", "disinflation",
]

LABOR_TERMS = [
    "employment", "unemployment", "labor market", "job gains", "payrolls",
    "hiring", "layoffs", "wages", "labor demand", "labor supply",
    "labor force", "slack",
]

pd.DataFrame({
    "inflation": pd.Series(INFLATION_TERMS),
    "labor": pd.Series(LABOR_TERMS),
})

In [ ]:
def count_terms(text, terms):
    text = clean_text(text)
    total = 0
    for term in terms:
        pattern = re.escape(term.lower()).replace(r"\ ", r"\s+")
        total += len(re.findall(rf"\b{pattern}\b", text))
    return total


def add_topic_score(data, corpus, topic, terms):
    text_col = f"{corpus}_clean_text"
    n_col = f"{corpus}_n_words"
    count_col = f"{corpus}_{topic}_count"
    freq_col = f"{corpus}_{topic}_per_1000"

    counts = data[text_col].map(lambda text: count_terms(text, terms))

    return data.assign(**{
        count_col: counts,
        freq_col: lambda x: 1_000 * x[count_col] / x[n_col].replace(0, np.nan),
    })


def period_label(date):
    date = pd.Timestamp(date)
    if pd.Timestamp("2001-03-01") <= date <= pd.Timestamp("2001-11-30"):
        return "2001 recession"
    if pd.Timestamp("2007-12-01") <= date <= pd.Timestamp("2009-06-30"):
        return "GFC"
    if pd.Timestamp("2020-02-01") <= date <= pd.Timestamp("2020-06-30"):
        return "Covid shock"
    if pd.Timestamp("2021-01-01") <= date <= pd.Timestamp("2024-12-31"):
        return "post-Covid inflation"
    return "other"


PERIOD_ORDER = ["2001 recession", "GFC", "Covid shock", "post-Covid inflation", "other"]

df = (
    df.sort_values("date")
    .reset_index(drop=True)
    .pipe(add_topic_score, "statement", "inflation", INFLATION_TERMS)
    .pipe(add_topic_score, "statement", "labor", LABOR_TERMS)
    .pipe(add_topic_score, "minutes", "inflation", INFLATION_TERMS)
    .pipe(add_topic_score, "minutes", "labor", LABOR_TERMS)
    .assign(
        statement_balance=lambda x: x["statement_inflation_per_1000"] - x["statement_labor_per_1000"],
        minutes_balance=lambda x: x["minutes_inflation_per_1000"] - x["minutes_labor_per_1000"],
        period=lambda x: x["date"].map(period_label),
        next_rate_decision=lambda x: x["rate_decision"].shift(-1),
    )
    .assign(
        minutes_minus_statement_balance=lambda x: x["minutes_balance"] - x["statement_balance"],
        period=lambda x: pd.Categorical(x["period"], categories=PERIOD_ORDER, ordered=True),
    )
)

topic_long = (
    df.loc[:, [
        "date", "year", "period", "rate_decision", "next_rate_decision",
        "statement_inflation_per_1000", "statement_labor_per_1000",
        "minutes_inflation_per_1000", "minutes_labor_per_1000",
    ]]
    .melt(
        id_vars=["date", "year", "period", "rate_decision", "next_rate_decision"],
        var_name="measure",
        value_name="mentions_per_1000",
    )
    .assign(
        document_type=lambda x: x["measure"].str.extract(r"^(statement|minutes)"),
        topic=lambda x: x["measure"].str.extract(r"_(inflation|labor)_"),
    )
    .drop(columns="measure")
)

topic_long.head()

Premier repère : mentions moyennes pour 1 000 mots.

In [ ]:
(
    topic_long
    .groupby(["document_type", "topic"], observed=True)["mentions_per_1000"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

In [ ]:
yearly_topics = (
    topic_long
    .groupby(["year", "document_type", "topic"], observed=True, as_index=False)["mentions_per_1000"]
    .mean()
)

g = sns.relplot(
    data=yearly_topics,
    x="year",
    y="mentions_per_1000",
    hue="topic",
    col="document_type",
    kind="line",
    height=3.5,
    aspect=1.4,
)
g.set_axis_labels("Year", "Mentions per 1,000 words")
g.set_titles("{col_name}");

L'inflation domine dans les deux corpus. Par 1 000 mots, les Statements sont plus concentrés que les Minutes sur les deux thèmes.

Score de balance : mentions d'inflation moins mentions du marché du travail, pour 1 000 mots.

Positif : le document parle davantage d'inflation.

In [ ]:
balance_long = (
    df.loc[:, [
        "date", "year", "period", "rate_decision", "next_rate_decision",
        "statement_balance", "minutes_balance",
    ]]
    .melt(
        id_vars=["date", "year", "period", "rate_decision", "next_rate_decision"],
        value_vars=["statement_balance", "minutes_balance"],
        var_name="document_type",
        value_name="balance_per_1000",
    )
    .assign(document_type=lambda x: x["document_type"].str.replace("_balance", "", regex=False))
)

(
    balance_long
    .groupby("document_type", observed=True)["balance_per_1000"]
    .describe()
    .round(2)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=balance_long, x="date", y="balance_per_1000", hue="document_type", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Inflation-labor balance")
ax.set_xlabel("Meeting date")
ax.set_ylabel("Inflation minus labor mentions");

Le score reste souvent positif : le vocabulaire d'inflation prend plus de place que celui du marché du travail.

Les dates où les deux formats divergent le plus.

In [ ]:
(
    df.assign(abs_gap=lambda x: x["minutes_minus_statement_balance"].abs())
    .nlargest(12, "abs_gap")
    .loc[:, [
        "date", "period", "rate_decision", "next_rate_decision",
        "statement_balance", "minutes_balance", "minutes_minus_statement_balance",
    ]]
    .round(2)
)

Lien rapide avec les décisions de taux.

Pour la corrélation, je code `cut=-1`, `hold=0`, `hike=1`.

In [ ]:
DECISION_CODE = {"cut": -1, "hold": 0, "hike": 1}

df = df.assign(
    rate_decision_code=lambda x: x["rate_decision"].map(DECISION_CODE),
    next_rate_decision_code=lambda x: x["next_rate_decision"].map(DECISION_CODE),
)

(
    df.loc[:, [
        "statement_balance", "minutes_balance",
        "rate_decision_code", "next_rate_decision_code",
    ]]
    .corr()
    .loc[["statement_balance", "minutes_balance"], ["rate_decision_code", "next_rate_decision_code"]]
    .round(2)
)

La corrélation est positive avec la décision courante et avec la décision suivante. Les réunions plus orientées inflation sont plus souvent proches de hausses de taux.

In [ ]:
(
    balance_long
    .groupby(["document_type", "rate_decision"], observed=True)["balance_per_1000"]
    .agg(["count", "mean", "median"])
    .round(2)
)

In [ ]:
(
    balance_long
    .dropna(subset=["next_rate_decision"])
    .groupby(["document_type", "next_rate_decision"], observed=True)["balance_per_1000"]
    .agg(["count", "mean", "median"])
    .round(2)
)

Les périodes sont des repères simples. Suffisant pour une première lecture.

In [ ]:
(
    topic_long
    .groupby(["period", "document_type", "topic"], observed=True)["mentions_per_1000"]
    .mean()
    .unstack("topic")
    .round(2)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(
    data=balance_long,
    x="period",
    y="balance_per_1000",
    hue="document_type",
    errorbar=None,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Inflation-labor balance by period")
ax.set_xlabel("")
ax.set_ylabel("Inflation minus labor mentions")
ax.tick_params(axis="x", rotation=25);

Covid ressort différemment : le marché du travail devient beaucoup plus visible dans les Statements que dans les crises précédentes.

Notes pour la prochaine lecture :

- relire les dates avec les plus gros écarts ;
- voir si les Minutes sont seulement plus détaillées ou vraiment orientées différemment ;
- comparer ensuite cette lecture par dictionnaire avec TF-IDF et embeddings.